## Анализ и валидация JSON-аннотаций

Этот ноутбук выполняет комплексную обработку JSON-аннотаций с результатами анализа изображений деревьев, включая валидацию данных и создание структурированного датасета:

### Основные компоненты:
- **JSON валидатор** - для проверки корректности структуры аннотаций
- **Pandas DataFrame** - для создания структурированного датасета
- **Статистический анализ** - для анализа распределения классов
- **Экспорт данных** - для сохранения в CSV и JSON форматах

### Процесс работы:
1. **Валидация структуры**: Проверяет соответствие JSON-файлов ожидаемой схеме
2. **Извлечение данных**: Парсит аннотации и извлекает метаданные
3. **Нормализация**: Приводит данные к единому формату с обработкой ошибок
4. **Создание датасета**: Формирует pandas DataFrame с полной информацией
5. **Анализ и экспорт**: Генерирует статистику и сохраняет результаты

### Возможности:
- **Валидация аннотаций**: Проверка структуры JSON, типов данных, обязательных полей
- **Извлечение метаданных**: 
  - Виды деревьев (50+ классов)
  - Болезни деревьев (20+ классов)
  - Координаты bounding box'ов
  - Оценки повреждений (дупла, трещины, гниль и т.д.)
- **Обработка ошибок**: Автоматическое исправление некорректных данных
- **Статистический анализ**: Распределение классов, частоты встречаемости

### Выходные данные:
- **CSV датасет**: `data/dataset.csv` с полной информацией об аннотациях

### Применение:
Идеально подходит для подготовки данных для обучения классификаторов видов и болезней деревьев, анализа качества аннотаций.

In [ ]:
JSON_DIR = "data/annotations"
OUT_CSV = "data/dataset.csv"
OUT_PARQUET = None
OUT_VOCABS = "data/label_vocabs.json"
OUT_REPORT = "data/validation_report.json"

In [ ]:
import os, json
from typing import List, Dict, Any, Tuple
import pandas as pd
from pathlib import Path
from typing import Tuple

In [ ]:
EXPECTED_ROOT_KEYS = {"img_path", "annotations", "model_name"}
EXPECTED_ANN_KEYS = {
    "bbox","class","species_probabilities","hollows","cracks","stem_damage",
    "crown_damage","fruiting_bodies","diseases","dry_branches_percentage"
}
EXPECTED_SPECIES_KEYS = {"species","probability"}
EXPECTED_DISEASE_KEYS = {"disease","probability"}

In [ ]:
def list_json_files(json_dir: str) -> List[str]:
    return sorted([os.path.join(json_dir, f) for f in os.listdir(json_dir)
                   if f.lower().endswith(".json") and os.path.isfile(os.path.join(json_dir, f))])

In [ ]:
def validate_and_collect_vocab(
    payload: Dict[str, Any], fname: str
) -> Tuple[List[str], List[str], List[str]]:
    errors: List[str] = []
    root_keys = set(payload.keys())
    missing = EXPECTED_ROOT_KEYS - root_keys
    extra   = root_keys - EXPECTED_ROOT_KEYS
    if missing:
        errors.append(f"{fname}: missing root keys: {sorted(missing)}")
    if extra:
        errors.append(f"{fname}: unexpected root keys: {sorted(extra)}")

    anns = payload.get("annotations", None)
    if not isinstance(anns, list):
        errors.append(f"{fname}: 'annotations' must be a list")
        return [], [], errors

    species_vocab = set()
    disease_vocab = set()

    for i, ann in enumerate(anns):
        if not isinstance(ann, dict):
            errors.append(f"{fname}: annotations[{i}] must be dict")
            continue
        ak = set(ann.keys())
        miss = EXPECTED_ANN_KEYS - ak
        ext  = ak - EXPECTED_ANN_KEYS
        if miss:
            errors.append(f"{fname}: annotations[{i}] missing keys: {sorted(miss)}")
        if ext:
            errors.append(f"{fname}: annotations[{i}] unexpected keys: {sorted(ext)}")

        bbox = ann.get("bbox")
        if not (isinstance(bbox, list) and len(bbox)==4 and all(isinstance(v,(int,float)) for v in bbox)):
            errors.append(f"{fname}: annotations[{i}].bbox must be [x1,y1,x2,y2] numbers")

        sp = ann.get("species_probabilities")
        if not (isinstance(sp, list) and len(sp)>=1):
            errors.append(f"{fname}: annotations[{i}].species_probabilities must be non-empty list")
        else:
            for j, item in enumerate(sp):
                if not isinstance(item, dict):
                    errors.append(f"{fname}: annotations[{i}].species_probabilities[{j}] must be dict")
                    continue
                sk = set(item.keys())
                if sk != EXPECTED_SPECIES_KEYS:
                    errors.append(f"{fname}: annotations[{i}].species_probabilities[{j}] keys must be {sorted(EXPECTED_SPECIES_KEYS)}, got {sorted(sk)}")
                else:
                    species = item.get("species")
                    prob = item.get("probability")
                    if not isinstance(species, str):
                        errors.append(f"{fname}: annotations[{i}].species_probabilities[{j}].species must be str")
                    if not isinstance(prob, (int,float)):
                        errors.append(f"{fname}: annotations[{i}].species_probabilities[{j}].probability must be number")
                    if isinstance(species, str):
                        species_vocab.add(species)

        dis = ann.get("diseases")
        if not isinstance(dis, list):
            errors.append(f"{fname}: annotations[{i}].diseases must be list")
        else:
            for j, item in enumerate(dis):
                if not isinstance(item, dict):
                    errors.append(f"{fname}: annotations[{i}].diseases[{j}] must be dict")
                    continue
                dk = set(item.keys())
                if dk != EXPECTED_DISEASE_KEYS:
                    errors.append(f"{fname}: annotations[{i}].diseases[{j}] keys must be {sorted(EXPECTED_DISEASE_KEYS)}, got {sorted(dk)}")
                else:
                    disease = item.get("disease")
                    prob = item.get("probability")
                    if not isinstance(disease, str):
                        errors.append(f"{fname}: annotations[{i}].diseases[{j}].disease must be str")
                    if not isinstance(prob, (int,float)):
                        errors.append(f"{fname}: annotations[{i}].diseases[{j}].probability must be number")
                    if isinstance(disease, str):
                        disease_vocab.add(disease)

        for fld in ["hollows","cracks","stem_damage","crown_damage","fruiting_bodies"]:
            v = ann.get(fld)
            if not isinstance(v, (int,float)):
                errors.append(f"{fname}: annotations[{i}].{fld} must be number in [0,1]")

        v = ann.get("dry_branches_percentage")
        if not isinstance(v, (int,float)):
            errors.append(f"{fname}: annotations[{i}].dry_branches_percentage must be number (0..100)")

        if not isinstance(ann.get("class", None), str):
            errors.append(f"{fname}: annotations[{i}].class must be str")

    return sorted(species_vocab), sorted(disease_vocab), errors

In [ ]:
def build_dataframe(files: List[str]) -> Tuple[pd.DataFrame, Dict[str, List[str]], Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    species_all: set = set()
    disease_all: set = set()
    all_errors: List[str] = []

    payloads: Dict[str, Dict[str, Any]] = {}
    for f in files:
        base = os.path.basename(f)
        try:
            with open(f, "r", encoding="utf-8") as fh:
                data = json.load(fh)
        except Exception as e:
            all_errors.append(f"{base}: JSON parse error: {e}")
            continue

        payloads[f] = data
        s_vocab, d_vocab, errors = validate_and_collect_vocab(data, base)
        species_all.update(s_vocab)
        disease_all.update(d_vocab)
        all_errors.extend(errors)

    species_list = sorted(species_all)
    disease_list = sorted(disease_all)

    for f, data in payloads.items():
        base = os.path.basename(f)
        img_path = data.get("img_path", None)
        model_name = data.get("model_name", None)

        anns = data.get("annotations")
        if not isinstance(anns, list):
            all_errors.append(f"{base}: 'annotations' is not a list (skipped)")
            continue

        for i, ann in enumerate(anns):
            bbox = ann.get("bbox")
            if not (isinstance(bbox, list) and len(bbox) == 4 and all(isinstance(v, (int, float)) for v in bbox)):
                all_errors.append(f"{base}: annotations[{i}].bbox is empty/invalid -> default [0,0,1,1]")
                bbox = [0, 0, 1, 1]
            bbox = [int(round(bbox[0])), int(round(bbox[1])), int(round(bbox[2])), int(round(bbox[3]))]

            cls = ann.get("class", "")
            if not isinstance(cls, str) or cls.strip() == "":
                all_errors.append(f"{base}: annotations[{i}].class is empty -> ''")
                cls = "" if not isinstance(cls, str) else cls

            sp_raw = ann.get("species_probabilities")
            if not isinstance(sp_raw, list):
                sp_raw = []
                all_errors.append(f"{base}: annotations[{i}].species_probabilities missing/invalid -> []")
            sp_norm: List[Dict[str, Any]] = []
            for j, item in enumerate(sp_raw):
                if not isinstance(item, dict):
                    all_errors.append(f"{base}: annotations[{i}].species_probabilities[{j}] not a dict -> skip")
                    continue
                species = item.get("species") if isinstance(item.get("species"), str) else ""
                try:
                    prob = float(item.get("probability", 0.0) or 0.0)
                except Exception:
                    prob = 0.0
                sp_norm.append({"species": species, "probability": prob})
            species_label = (max(sp_norm, key=lambda x: x["probability"])["species"] if sp_norm else "")

            dis_raw = ann.get("diseases")
            if not isinstance(dis_raw, list):
                dis_raw = []
                all_errors.append(f"{base}: annotations[{i}].diseases missing/invalid -> []")
            dis_norm: List[Dict[str, Any]] = []
            for j, item in enumerate(dis_raw):
                if not isinstance(item, dict):
                    all_errors.append(f"{base}: annotations[{i}].diseases[{j}] not a dict -> skip")
                    continue
                disease = item.get("disease") if isinstance(item.get("disease"), str) else ""
                try:
                    prob = float(item.get("probability", 0.0) or 0.0)
                except Exception:
                    prob = 0.0
                dis_norm.append({"disease": disease, "probability": prob})
            disease_label = (max(dis_norm, key=lambda x: x["probability"])["disease"] if dis_norm else "")

            def _num01(val, field: str) -> float:
                if not isinstance(val, (int, float)):
                    all_errors.append(f"{base}: annotations[{i}].{field} is empty/invalid -> 0.0")
                    return 0.0
                return float(val)

            hollows = _num01(ann.get("hollows", 0.0), "hollows")
            cracks = _num01(ann.get("cracks", 0.0), "cracks")
            stem_damage = _num01(ann.get("stem_damage", 0.0), "stem_damage")
            crown_damage = _num01(ann.get("crown_damage", 0.0), "crown_damage")
            fruiting_bodies = _num01(ann.get("fruiting_bodies", 0.0), "fruiting_bodies")

            dry_pct = ann.get("dry_branches_percentage", 0.0)
            if not isinstance(dry_pct, (int, float)):
                all_errors.append(f"{base}: annotations[{i}].dry_branches_percentage is empty/invalid -> 0.0")
                dry_pct = 0.0
            dry_branches_frac = float(dry_pct) / 100.0

            rows.append({
                "base_name": base,
                "model_name": model_name,
                "class": cls,
                "bbox": bbox,
                "species_label": species_label,
                "disease_label": disease_label,
                "hollows": hollows,
                "cracks": cracks,
                "stem_damage": stem_damage,
                "crown_damage": crown_damage,
                "fruiting_bodies": fruiting_bodies,
                "dry_branches_frac": dry_branches_frac,
            })

    df = pd.DataFrame(rows)
    vocabs = {"species": species_list, "diseases": disease_list}
    report = {"errors": all_errors, "num_files": len(files), "num_rows": len(rows)}
    return df, vocabs, report

In [ ]:
def argmax_label_prob(
    items: List[Dict[str, Any]],
    name_key: str,
    prob_key: str
) -> Tuple[str, float]:
    if not items:
        return "", 0.0
    best = max(
        items,
        key=lambda x: (x.get(prob_key, 0.0) if isinstance(x.get(prob_key, None),(int,float)) else 0.0)
    )
    return best.get(name_key, ""), float(best.get(prob_key, 0.0) or 0.0)

In [ ]:
files = list_json_files(JSON_DIR)

In [ ]:
df, vocabs, report = build_dataframe(files)
df.head()

In [ ]:
report

In [ ]:
species = df["species_label"].unique()
len(species)

In [ ]:
species

In [ ]:
diseases = df["disease_label"].unique()
len(diseases)

In [ ]:
diseases

In [ ]:
df_csv = df.copy()
df_csv["bbox"] = df_csv["bbox"].apply(lambda b: json.dumps(b, ensure_ascii=False))
csv_path = "data/dataset.csv"
df_csv.to_csv(csv_path, index=False, encoding="utf-8")
print(f"CSV сохранён: {csv_path}")

In [ ]:
counts = df["disease_label"].astype(str).value_counts()
share  = counts / counts.sum()

dist = (
    pd.DataFrame({"count": counts, "share": share})
    .sort_values("count", ascending=False)
)

print(f"Disease_label classes: {len(dist)}")
dist.head(20)